# Final report figures

Generate report figures from the processed dataset and saved experiment results;
no models are trained. Run all cells in the project environment from any directory
inside the repository. Figures are displayed inline and saved as **PNG (200 dpi)** in `data/final_report/`; reruns replace files with the same names.

The notebook produces a one-week data example, a temporal evaluation diagram,
and **three monthly actual-versus-forecast figures**, each with three vertically
stacked panels for the best models in its scenario, ranked by **full-test MAE**, including the seasonal baseline among target-only
candidates. Hyperparameters were selected on validation; the top-three display
selection is descriptive and uses test results.
September 2026 is the first full calendar month of the common test period.
The monthly curves concatenate successive weekly forecasts; they are not a
single month-ahead forecast. All times are UTC.

Generation scenarios use future **actual generation as a perfect forecast**.
These results do not measure performance with real generation forecast errors.

In [ ]:
import json
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "docs/initial_assignment.md").exists()
)
RESULTS_DIR = ROOT / "experiments/results"
OUTPUT_DIR = ROOT / "data/final_report"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MONTH_START = pd.Timestamp("2026-09-01", tz="UTC")
MONTH_END = MONTH_START + pd.offsets.MonthBegin(1)  # Exclusive.
EXAMPLE_START = MONTH_START
EXAMPLE_END = EXAMPLE_START + pd.Timedelta(days=7)
SCENARIOS = {
    "target_only_v1": "Target-only",
    "generation_v1": "Full generation",
    "renewable_v1": "Renewable-only",
}
MODEL_NAMES = {
    "sma": "Seasonal moving average", "regression": "Linear regression",
    "random_forest": "Random forest", "lgbm": "LightGBM",
    "xgbm": "XGBoost", "catboost": "CatBoost",
}

plt.rcParams.update({"font.size": 10, "axes.titlesize": 12})


def save_and_show(fig, name):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{name}.png", dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def format_dates(ax):
    locator = mdates.AutoDateLocator(minticks=5, maxticks=9)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.grid(True, alpha=0.25)

## Load data and choose models

Read per-run metrics directly, so the combined metrics CSV need not be rebuilt.
First identify the lowest-validation-MAE run for each model and check that its
saved test run has the same parameters and inputs. Then rank the frozen models
within each scenario by full-test MAE and retain the top three for plotting.
The table reports **full-split** scores, not September-only scores.

In [ ]:
dataset = pd.read_csv(ROOT / "data/processed/dataset_v1.csv")
dataset["validfrom (UTC)"] = pd.to_datetime(dataset["validfrom (UTC)"], utc=True)
dataset = dataset.set_index("validfrom (UTC)").sort_index()

selected = []
for prefix, scenario in SCENARIOS.items():
    candidates = []
    for folder in sorted((RESULTS_DIR / "valid").glob(f"{prefix}_*")):
        runs = []
        for metric_path in sorted(folder.glob("*/metrics.json")):
            metrics = json.loads(metric_path.read_text())
            runs.append((metrics["mae_g_co2_per_kwh"], metric_path.parent, metrics))
        if runs:
            candidates.append(min(runs, key=lambda item: item[0]))
    assert len(candidates) >= 3, f"Need at least three validation models: {scenario}"
    for valid_mae, valid_path, valid_metrics in candidates:
        test_runs = sorted((RESULTS_DIR / "test" / valid_path.parent.name).glob("*/metrics.json"))
        assert len(test_runs) == 1, "Expected one frozen test configuration per model"
        test_path = test_runs[0].parent
        test_metrics = json.loads(test_runs[0].read_text())
        assert json.loads((valid_path / "params.json").read_text()) == json.loads(
            (test_path / "params.json").read_text()
        ), f"Validation/test parameter mismatch: {test_path}"
        for key in ("feature_columns", "target", "target_offset_hours"):
            assert valid_metrics[key] == test_metrics[key]
        selected.append({
            "Scenario": scenario,
            "Model": MODEL_NAMES[test_metrics["model"]],
            "Validation MAE": valid_mae,
            "Test MAE": test_metrics["mae_g_co2_per_kwh"],
            "Test RMSE": test_metrics["rmse_g_co2_per_kwh"],
            "Test R²": test_metrics["r2"],
            "prefix": prefix, "path": test_path,
        })

selected = [
    {**row, "Rank": rank}
    for scenario in SCENARIOS.values()
    for rank, row in enumerate(
        sorted(
            (row for row in selected if row["Scenario"] == scenario),
            key=lambda row: row["Test MAE"],
        )[:3], start=1
    )
]
selection = pd.DataFrame(selected)
display(selection.drop(columns=["prefix", "path"]).round(3))

## 1. One week of observed data

Show the target and all seven generation inputs on separate axes. Convert the
stored target from kg CO₂/kWh to g CO₂/kWh and hourly energy from kWh to GWh.
These are processed observations; the example is not a generation forecast.

In [ ]:
example = dataset.loc[(dataset.index >= EXAMPLE_START) & (dataset.index < EXAMPLE_END)]
assert len(example) == 168
sources = {
    "sun": "Solar", "wind_land": "Onshore wind", "wind_sea": "Offshore wind",
    "cogeneration": "Cogeneration", "biomass": "Biomass",
    "coal": "Coal", "natural_gas": "Natural gas",
}
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(example.index, example["y"] * 1000, color="tab:blue", linewidth=1)
axes[0].set(title="Observed data — first test week", ylabel="Carbon intensity (g CO₂/kWh)")
for source, label in sources.items():
    axes[1].plot(example.index, example[f"{source}_volume (kWh)"] / 1e6,
                 label=label, linewidth=1)
axes[1].set(xlabel="Time (UTC)", ylabel="Hourly generation (GWh)")
axes[1].legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.2))
for ax in axes:
    format_dates(ax)
save_and_show(fig, "01_observed_data_week")

## 2. Chronological evaluation

Split boundaries refer to **target time**. Hyperparameters are selected on
validation and frozen for test. At each weekly origin the model is refitted on
all labels available at that origin, including earlier validation/test labels.
Only complete 168-hour windows are evaluated. The lower panel illustrates the
first three test windows; the displayed history extends further into the past.
For target-only, the 168 historical feature rows ending at the origin map to
the next 168 target hours using the fixed 168-hour offset.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={"height_ratios": [1, 2]})
for split, color, label in (
    ("train", "#b9d8ef", "Train"), ("valid", "#f5d78b", "Validation"),
    ("test", "#a8d8b4", "Test"),
):
    times = dataset.index[dataset["split"].eq(split)]
    start, end = times.min(), times.max() + pd.Timedelta(hours=1)
    axes[0].axvspan(start, end, color=color)
    axes[0].text(start + (end - start) / 2, 0.5, label, ha="center", va="center")
axes[0].set(title="Chronological splits by target time", yticks=[], xlabel="Target time (UTC)")
format_dates(axes[0])

history_start = MONTH_START - pd.Timedelta(days=7)
for step in range(3):
    start = MONTH_START + pd.Timedelta(days=7 * step)
    end = start + pd.Timedelta(days=7)
    origin = start - pd.Timedelta(hours=1)
    y = 2 - step
    axes[1].broken_barh(
        [(mdates.date2num(history_start), (origin - history_start).total_seconds() / 86400)],
        (y - 0.22, 0.44), facecolors="#b9d8ef",
        label="Available labels for refitting (history continues left)" if step == 0 else None,
    )
    axes[1].broken_barh(
        [(mdates.date2num(start), 7)], (y - 0.22, 0.44), facecolors="#a8d8b4",
        label="168 target hours" if step == 0 else None,
    )
    axes[1].plot(origin, y, "ko", markersize=4,
                 label="Forecast origin / refit" if step == 0 else None)
    axes[1].text(start + pd.Timedelta(days=3.5), y, "168 h", ha="center", va="center")
axes[1].set(
    title="Weekly expanding evaluation — first three test windows",
    yticks=[2, 1, 0], yticklabels=["Week 1", "Week 2", "Week 3"], xlabel="Target time (UTC)",
)
axes[1].legend(loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2)
format_dates(axes[1])
save_and_show(fig, "02_evaluation_protocol")

## 3. Monthly forecasts for the top three models in each scenario

Each figure follows the line-plot style of `prediction_analysis.ipynb`: actual
and forecast at hourly target times, without smoothing or interpolation.
Each scenario has one figure with three model panels, ordered by full-test MAE.
All panels use identical time and target limits. Dotted vertical lines
mark the starts of new weekly forecast windows. Each panel title gives **September-only**
MAE and RMSE, recomputed from the displayed 720 hours; the table above retains
full-test metrics. The top-three display selection uses full-test MAE, not these monthly scores;
hyperparameters remain selected on validation.

In [ ]:
expected_times = pd.date_range(MONTH_START, MONTH_END, freq="h", inclusive="left")
monthly_predictions = {}
reference = None
for row in selected:
    frame = pd.read_csv(row["path"] / "predictions.csv")
    frame["target_time (UTC)"] = pd.to_datetime(frame["target_time (UTC)"], utc=True)
    frame = frame.set_index("target_time (UTC)").sort_index()
    month = frame.loc[(frame.index >= MONTH_START) & (frame.index < MONTH_END)].copy()
    assert month.index.equals(expected_times), "Monthly coverage must be complete and unique"
    assert month[["y", "y_pred"]].notna().all().all()
    if reference is None:
        reference = month["y"]
    else:
        pd.testing.assert_series_equal(month["y"], reference)
    monthly_predictions[row["path"]] = month

lower = min(frame[["y", "y_pred"]].min().min() for frame in monthly_predictions.values()) * 1000
upper = max(frame[["y", "y_pred"]].max().max() for frame in monthly_predictions.values()) * 1000
padding = (upper - lower) * 0.05


def plot_scenario(prefix):
    rows = [row for row in selected if row["prefix"] == prefix]
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True, sharey=True)
    fig.suptitle(f"{SCENARIOS[prefix]} — {MONTH_START:%B %Y}", fontsize=14)
    for ax, row in zip(axes, rows, strict=True):
        month = monthly_predictions[row["path"]]
        error = (month["y_pred"] - month["y"]) * 1000
        mae, rmse = error.abs().mean(), (error.pow(2).mean()) ** 0.5
        ax.plot(month.index, month["y"] * 1000, label="Actual", linewidth=1)
        ax.plot(month.index, month["y_pred"] * 1000, label="Forecast", linewidth=1)
        window_starts = month.groupby("forecast_origin (UTC)").apply(lambda group: group.index.min())
        for start in window_starts.iloc[1:]:
            ax.axvline(start, color="gray", linestyle=":", linewidth=0.8, alpha=0.5)
        ax.set(
            title=f"{row['Rank']}. {row['Model']} | Monthly MAE: {mae:.2f} | "
                  f"RMSE: {rmse:.2f} g CO₂/kWh",
            ylabel="Carbon intensity (g CO₂/kWh)",
            xlim=(MONTH_START, MONTH_END), ylim=(lower - padding, upper + padding),
        )
        ax.legend()
        format_dates(ax)
    axes[-1].set_xlabel("Target time (UTC)")
    save_and_show(fig, f"03_{prefix}_{MONTH_START:%Y_%m}")

### Target-only

In [ ]:
plot_scenario("target_only_v1")

### Full generation

In [ ]:
plot_scenario("generation_v1")

### Renewable-only

In [ ]:
plot_scenario("renewable_v1")

## Saved figures

The output folder contains five PNG files: one observed-data figure, one
evaluation diagram, and three monthly prediction figures with three model panels each. Files from previous
runs with different settings are not removed automatically.

In [ ]:
print(f"Figures saved to: {OUTPUT_DIR.relative_to(ROOT)}")
for path in sorted(OUTPUT_DIR.glob("*.png")):
    print(path.name)